
# Capítulo 4 — Modelagem passo a passo, sem “caixa-preta”

Este notebook foi preparado para continuar o TCC a partir do ponto em que o notebook `modeling_step_by_step.ipynb` parou.

## Objetivo

Construir, entender e avaliar uma cadeia experimental completa:

1. definir **o que estamos prevendo e em que momento a previsão acontece**;
2. evitar vazamento temporal (*data leakage*);
3. criar baselines históricos fortes;
4. treinar modelos de Machine Learning para a permanência total no porto;
5. avaliar o erro geral e a cauda da distribuição;
6. estimar quantis de risco (P50, P90 e P95);
7. traduzir os quantis em um **exercício de estoque de segurança / capital de giro**;
8. gerar tabelas e gráficos que possam ser usados no Capítulo 4.

> **Princípio central:** em cada etapa, rode a célula, observe a saída e só avance quando conseguir explicar em palavras o que aconteceu.



## 0. Antes de modelar: qual é o instante da previsão?

Para este TCC, a interpretação mais coerente é:

> **No instante em que a embarcação chega ao porto (`arrival_port_ts`), queremos estimar quanto tempo ela permanecerá no porto.**

Isso define a regra mais importante do experimento:

### Uma feature só pode ser usada se estivesse disponível no momento da chegada.

Consequências práticas:

- `port`, região, estado, calendário e tipo de operação: em princípio podem ser conhecidos na chegada;
- variáveis climáticas **históricas** (1, 3 e 7 dias anteriores): são seguras;
- clima realizado do próprio dia completo pode conter informação posterior ao horário de chegada;
- `arrivals_same_day_port` contém chegadas posteriores no mesmo dia e, portanto, pode vazar futuro;
- médias de duração das “20 chamadas anteriores por ordem de chegada” também merecem cautela: uma chamada que chegou antes pode ainda não ter terminado quando a chamada atual chega.

Por isso, começaremos com um **conjunto conservador de features sem vazamento evidente**. Depois, se necessário, reconstruiremos as features operacionais de forma mais rigorosa.


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    mean_pinball_loss,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..")
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "eda_base.parquet"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "t_total_port_stay_h"
DATE_COL = "arrival_port_ts"

df = pd.read_parquet(DATA_FILE)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, TARGET]).copy()

print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]:,}")
print(f"Período: {df[DATE_COL].min()} até {df[DATE_COL].max()}")



### 0.1 Faça uma checagem rápida da variável-alvo

O ponto principal é lembrar por que a mediana é uma referência importante:

- a média é puxada para cima pelos casos extremos;
- a mediana representa melhor o comportamento típico;
- RMSE muito maior que MAE é um sinal de que erros extremos importam.


In [ ]:

target_summary = df[TARGET].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
target_summary



## 1. Feature engineering seguro

### 1.1 Variáveis cíclicas

Hora, dia da semana e mês são variáveis circulares.  
23h está perto de 0h, e dezembro está perto de janeiro.

Em vez de deixar o modelo interpretar esses números como uma linha reta, criamos seno e cosseno.


In [ ]:

df["arrival_hour_sin"] = np.sin(2 * np.pi * df["arrival_hour"] / 24)
df["arrival_hour_cos"] = np.cos(2 * np.pi * df["arrival_hour"] / 24)

df["arrival_dow_sin"] = np.sin(2 * np.pi * df["arrival_dayofweek"] / 7)
df["arrival_dow_cos"] = np.cos(2 * np.pi * df["arrival_dayofweek"] / 7)

df["arrival_month_sin"] = np.sin(2 * np.pi * (df["arrival_month"] - 1) / 12)
df["arrival_month_cos"] = np.cos(2 * np.pi * (df["arrival_month"] - 1) / 12)



### 1.2 Conjunto conservador de features

Nesta primeira versão **não usaremos**:

- `arrivals_same_day_port`;
- clima observado do próprio dia;
- `avg_wait_prev_20_calls_port`;
- `avg_operation_prev_20_calls_port`;
- `std_wait_prev_20_calls_port`.

Essas variáveis podem ser úteis, mas precisam ser reconstruídas ou justificadas conforme o instante da previsão.


In [ ]:

CATEGORICAL_FEATURES = [
    "port",
    "region",
    "state",
    "arrival_shift",
    "arrival_season",
]

NUMERICAL_FEATURES = [
    "arrival_quarter",
    "arrival_weekofyear",
    "arrival_is_weekend",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "arrival_dow_sin",
    "arrival_dow_cos",
    "arrival_month_sin",
    "arrival_month_cos",

    # Congestionamento conhecido antes da chegada atual
    "arrivals_prev_day_port",
    "arrivals_prev_7d_avg_port",

    # Clima histórico
    "rain_sum_prev_1d",
    "precipitation_sum_prev_1d",
    "wind_speed_10m_max_prev_1d",
    "wind_gusts_10m_max_prev_1d",
    "temperature_2m_mean_prev_1d",
    "rain_sum_prev_3d",
    "precipitation_hours_prev_3d",
    "temperature_2m_mean_prev_3d",
    "wind_speed_10m_max_prev_3d",
    "wind_gusts_10m_max_prev_3d",
    "rain_sum_prev_7d",
    "precipitation_hours_prev_7d",
    "temperature_2m_mean_prev_7d",
    "wind_speed_10m_max_prev_7d",
    "wind_gusts_10m_max_prev_7d",
]

OPERATION_FEATURES = sorted(
    c for c in df.columns if c.startswith("op_")
)

CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in df.columns]
NUMERICAL_FEATURES = [c for c in NUMERICAL_FEATURES if c in df.columns]
OPERATION_FEATURES = [c for c in OPERATION_FEATURES if c in df.columns]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES + OPERATION_FEATURES

print(f"Categóricas: {len(CATEGORICAL_FEATURES)}")
print(f"Numéricas: {len(NUMERICAL_FEATURES)}")
print(f"Flags de operação: {len(OPERATION_FEATURES)}")
print(f"Total de features antes do One-Hot: {len(FEATURES)}")



## 2. Divisão temporal

O notebook anterior usou:

- treino: até 30/06/2024;
- validação: 01/07/2024 a 31/12/2024;
- teste: 2025.

Como os resultados de 2025 **já foram consultados nos baselines**, o conjunto de 2025 não é mais um holdout totalmente “cego”.

Para aumentar o rigor, esta versão separa:

- **Treino:** 2023 até 30/06/2024;
- **Validação:** 01/07/2024 a 31/12/2024;
- **Calibração / desenvolvimento:** 01/01/2025 a 30/06/2025;
- **Teste final:** 01/07/2025 a 31/12/2025.

### Como usar

- treino + validação: escolha de modelo;
- calibração: checagem de quantis e decisões finais;
- teste final: abrir apenas quando a metodologia estiver congelada.

> Se você decidir manter o desenho original do texto, pode voltar a usar 2025 inteiro como teste, mas registre essa limitação metodológica.


In [ ]:

train_df = df[df[DATE_COL] < "2024-07-01"].copy()

val_df = df[
    (df[DATE_COL] >= "2024-07-01")
    & (df[DATE_COL] < "2025-01-01")
].copy()

cal_df = df[
    (df[DATE_COL] >= "2025-01-01")
    & (df[DATE_COL] < "2025-07-01")
].copy()

test_df = df[df[DATE_COL] >= "2025-07-01"].copy()

split_summary = pd.DataFrame({
    "dataset": ["train", "validation", "calibration", "final_test"],
    "rows": [len(train_df), len(val_df), len(cal_df), len(test_df)],
    "min_date": [
        train_df[DATE_COL].min(),
        val_df[DATE_COL].min(),
        cal_df[DATE_COL].min(),
        test_df[DATE_COL].min(),
    ],
    "max_date": [
        train_df[DATE_COL].max(),
        val_df[DATE_COL].max(),
        cal_df[DATE_COL].max(),
        test_df[DATE_COL].max(),
    ],
})
split_summary



## 3. Funções de avaliação

Para regressão de permanência:

- **MAE:** erro médio em horas; será a métrica principal;
- **RMSE:** dá peso maior a erros muito grandes;
- **MedAE:** mostra o erro do caso “típico”;
- **RMSLE:** compara erros em escala logarítmica e reduz a dominância de valores extremos.


In [ ]:

def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true), 0, None)
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def regression_metrics(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "medae": median_absolute_error(y_true, y_pred),
        "rmsle": rmsle(y_true, y_pred),
    }

def add_result(results, model_name, dataset_name, y_true, y_pred):
    row = {
        "model": model_name,
        "dataset": dataset_name,
        **regression_metrics(y_true, y_pred),
    }
    results.append(row)
    return row



## 4. Baselines históricos

### 4.1 Mediana global

Pergunta respondida:

> “Se eu ignorar todo o contexto, qual é o tempo típico histórico?”


In [ ]:

results = []

global_median = train_df[TARGET].median()
pred = np.repeat(global_median, len(val_df))

add_result(
    results,
    "baseline_global_median",
    "validation",
    val_df[TARGET],
    pred,
)

pd.DataFrame(results)



### 4.2 Mediana por porto

Pergunta:

> “Dado o porto em que a embarcação chegou, qual foi a permanência típica histórica nesse porto?”


In [ ]:

def group_median_prediction(train, scoring, group_cols, min_group_size=1):
    global_median = train[TARGET].median()

    stats = (
        train.groupby(group_cols, dropna=False)[TARGET]
        .agg(["median", "count"])
        .reset_index()
    )

    stats.loc[stats["count"] < min_group_size, "median"] = np.nan

    scored = scoring[group_cols].copy()
    scored = scored.merge(
        stats[group_cols + ["median"]],
        on=group_cols,
        how="left",
    )

    return scored["median"].fillna(global_median).to_numpy()

pred_port = group_median_prediction(
    train_df,
    val_df,
    ["port"],
)

add_result(
    results,
    "baseline_median_by_port",
    "validation",
    val_df[TARGET],
    pred_port,
)

pd.DataFrame(results)



### 4.3 Baseline hierárquico: porto + combinação de operação → porto → global

No notebook anterior, 73% dos grupos `porto + operation_type` tinham menos de 30 observações no treino.

Isso significa que aceitar qualquer combinação, mesmo com 1 ou 2 exemplos, pode produzir uma “mediana” instável.

A regra hierárquica é:

1. usar `porto + operation_type` somente se o grupo tiver pelo menos 30 casos;
2. senão, usar a mediana do porto;
3. se o porto também não existir, usar a mediana global.


In [ ]:

def hierarchical_median_prediction(
    train,
    scoring,
    primary_cols,
    fallback_cols,
    min_group_size=30,
):
    global_median = train[TARGET].median()

    primary = (
        train.groupby(primary_cols, dropna=False)[TARGET]
        .agg(["median", "count"])
        .reset_index()
    )
    primary = primary[primary["count"] >= min_group_size]
    primary = primary[primary_cols + ["median"]].rename(
        columns={"median": "primary_median"}
    )

    fallback = (
        train.groupby(fallback_cols, dropna=False)[TARGET]
        .agg(["median", "count"])
        .reset_index()
    )
    fallback = fallback[fallback["count"] >= min_group_size]
    fallback = fallback[fallback_cols + ["median"]].rename(
        columns={"median": "fallback_median"}
    )

    out = scoring[primary_cols].copy()
    out = out.merge(primary, on=primary_cols, how="left")
    out = out.merge(fallback, on=fallback_cols, how="left")

    return (
        out["primary_median"]
        .fillna(out["fallback_median"])
        .fillna(global_median)
        .to_numpy()
    )

pred_hier = hierarchical_median_prediction(
    train_df,
    val_df,
    primary_cols=["port", "operation_type"],
    fallback_cols=["port"],
    min_group_size=30,
)

add_result(
    results,
    "baseline_hierarchical_port_operation",
    "validation",
    val_df[TARGET],
    pred_hier,
)

baseline_results = pd.DataFrame(results)
baseline_results["mae_improvement_vs_global_pct"] = (
    (baseline_results.loc[0, "mae"] - baseline_results["mae"])
    / baseline_results.loc[0, "mae"]
    * 100
)
baseline_results



### O que você deve conseguir explicar antes de seguir

- Por que a mediana global é um baseline?
- Por que segmentar por porto melhora a previsão?
- Por que `porto + operação` pode melhorar ainda mais?
- Por que grupos muito pequenos precisam de fallback?
- Por que estamos avaliando primeiro na validação, e não no teste final?



## 5. Pré-processamento para Machine Learning

Temos dois tipos principais de variável:

- categóricas → preenchimento + One-Hot Encoding;
- numéricas → preenchimento pela mediana.

Para o Ridge, também aplicaremos `StandardScaler`, porque a regularização depende da escala das variáveis.

Para modelos de árvore, a escala numérica não é necessária.


In [ ]:

def build_preprocessor(scale_numeric=False):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, NUMERICAL_FEATURES + OPERATION_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ])

X_train = train_df[FEATURES].copy()
X_val = val_df[FEATURES].copy()

y_train = train_df[TARGET].copy()
y_val = val_df[TARGET].copy()



## 6. Modelo 1 — Ridge na escala logarítmica

Por que log?

A permanência é positiva, assimétrica e possui cauda longa.  
Modelar `log(1 + permanência)` reduz a influência dos extremos durante o ajuste.

Ao final usamos `expm1` para voltar a horas.


In [ ]:

ridge = Pipeline([
    ("preprocessor", build_preprocessor(scale_numeric=True)),
    ("model", Ridge(alpha=1.0)),
])

ridge.fit(X_train, np.log1p(y_train))

pred_ridge_val = np.expm1(ridge.predict(X_val))
pred_ridge_val = np.clip(pred_ridge_val, 0, None)

add_result(
    results,
    "ridge_log_target",
    "validation",
    y_val,
    pred_ridge_val,
)

pd.DataFrame(results).sort_values("mae")



## 7. Modelo 2 — Random Forest

A Random Forest consegue representar interações não lineares, por exemplo:

- um mesmo volume de chuva pode ter efeitos diferentes por porto;
- um mesmo tipo de operação pode ter tempos diferentes entre regiões;
- efeitos de calendário e congestionamento podem interagir.

Ela não exige que essas relações sejam especificadas manualmente.


In [ ]:

rf = Pipeline([
    ("preprocessor", build_preprocessor(scale_numeric=False)),
    ("model", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

rf.fit(X_train, y_train)

pred_rf_val = np.clip(rf.predict(X_val), 0, None)

add_result(
    results,
    "random_forest",
    "validation",
    y_val,
    pred_rf_val,
)

pd.DataFrame(results).sort_values("mae")



## 8. Modelo 3 — Gradient Boosting

O Gradient Boosting constrói árvores sequencialmente, fazendo cada nova árvore tentar corrigir erros das anteriores.

Esta célula usa o `GradientBoostingRegressor` do scikit-learn para permanecer consistente com o código atual do repositório.

> Dependendo da máquina, esta etapa pode levar alguns minutos.


In [ ]:

gbr = Pipeline([
    ("preprocessor", build_preprocessor(scale_numeric=False)),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    )),
])

gbr.fit(X_train, y_train)

pred_gbr_val = np.clip(gbr.predict(X_val), 0, None)

add_result(
    results,
    "gradient_boosting",
    "validation",
    y_val,
    pred_gbr_val,
)

model_comparison = pd.DataFrame(results).sort_values("mae").reset_index(drop=True)
model_comparison



### 8.1 Não escolha o modelo olhando apenas uma métrica

Antes de decidir:

1. compare MAE;
2. compare RMSE — o modelo explode nos extremos?
3. compare MedAE — ele melhora o caso típico?
4. compare RMSLE — como se comporta proporcionalmente?
5. compare o ganho contra o **melhor baseline**, não apenas contra a mediana global.


In [ ]:

best_baseline_mae = (
    model_comparison[
        model_comparison["model"].str.startswith("baseline")
    ]["mae"].min()
)

model_comparison["mae_improvement_vs_best_baseline_pct"] = (
    (best_baseline_mae - model_comparison["mae"])
    / best_baseline_mae
    * 100
)

model_comparison



## 9. Avaliação segmentada

Um modelo pode ter um MAE agregado bom e ainda ser ruim nos portos críticos.

Vamos analisar:

- região;
- porto;
- faixas reais de permanência;
- flags de operação.


In [ ]:

# Escolha o modelo a investigar.
# Troque esta variável se outro modelo for melhor na validação.
analysis_model_name = "gradient_boosting"
analysis_pred = pred_gbr_val

val_eval = val_df[
    ["port", "region", "state", "operation_type", TARGET] + OPERATION_FEATURES
].copy()
val_eval["prediction"] = analysis_pred
val_eval["abs_error"] = np.abs(val_eval[TARGET] - val_eval["prediction"])

val_eval["target_range"] = pd.cut(
    val_eval[TARGET],
    bins=[0, 24, 48, 72, 168, np.inf],
    labels=[
        "até 1 dia",
        "1 a 2 dias",
        "2 a 3 dias",
        "3 a 7 dias",
        "acima de 7 dias",
    ],
    include_lowest=True,
)

segment_by_region = (
    val_eval.groupby("region", dropna=False)
    .agg(
        rows=(TARGET, "size"),
        mae=("abs_error", "mean"),
        actual_median=(TARGET, "median"),
        predicted_median=("prediction", "median"),
    )
    .sort_values("mae", ascending=False)
)

segment_by_region


In [ ]:

segment_by_target = (
    val_eval.groupby("target_range", observed=False)
    .agg(
        rows=(TARGET, "size"),
        mae=("abs_error", "mean"),
        actual_median=(TARGET, "median"),
        predicted_median=("prediction", "median"),
    )
)

segment_by_target



### Interpretação essencial

Se o MAE crescer muito na faixa “acima de 7 dias”, isso confirma algo importante para o TCC:

> o desafio principal não é somente prever a mediana da permanência; é capturar corretamente a **cauda de risco**.

Isso justifica a próxima etapa: regressão quantílica.



## 10. Regressão quantílica — P50, P90 e P95

Um modelo convencional tenta produzir um único número.

Um modelo quantílico responde perguntas diferentes:

- P50: “há 50% de chance de a permanência ficar abaixo deste valor”;
- P90: “há 90% de chance de ficar abaixo deste valor”;
- P95: cenário ainda mais conservador.

Para Supply Chain, P90/P95 são úteis porque representam explicitamente o risco de atraso.


In [ ]:

QUANTILES = [0.50, 0.90, 0.95]

quantile_models = {}
quantile_predictions_cal = {}

# Reajustamos com treino + validação antes da calibração.
dev_df = pd.concat([train_df, val_df], ignore_index=True)
X_dev = dev_df[FEATURES]
y_dev = dev_df[TARGET]

X_cal = cal_df[FEATURES]
y_cal = cal_df[TARGET]

for q in QUANTILES:
    model = Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("model", GradientBoostingRegressor(
            loss="quantile",
            alpha=q,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        )),
    ])

    print(f"Treinando quantil {q:.2f}...")
    model.fit(X_dev, y_dev)

    pred = np.clip(model.predict(X_cal), 0, None)

    quantile_models[q] = model
    quantile_predictions_cal[q] = pred

print("Concluído.")



### 10.1 Como avaliar um quantil

Não devemos avaliar P90 somente com MAE.

As duas perguntas principais são:

1. **Pinball loss:** o erro específico para aquele quantil;
2. **Cobertura empírica:** aproximadamente 90% dos valores reais deveriam ficar abaixo do P90 previsto.

Um P90 que cobre apenas 75% está otimista.  
Um P90 que cobre 99% pode estar conservador demais.


In [ ]:

quantile_eval_rows = []

for q in QUANTILES:
    pred = quantile_predictions_cal[q]

    quantile_eval_rows.append({
        "quantile": q,
        "pinball_loss": mean_pinball_loss(y_cal, pred, alpha=q),
        "empirical_coverage": np.mean(y_cal.to_numpy() <= pred),
        "coverage_error": np.mean(y_cal.to_numpy() <= pred) - q,
    })

quantile_eval = pd.DataFrame(quantile_eval_rows)
quantile_eval


In [ ]:

q50 = quantile_predictions_cal[0.50]
q90 = quantile_predictions_cal[0.90]
q95 = quantile_predictions_cal[0.95]

crossing_50_90 = np.mean(q50 > q90)
crossing_90_95 = np.mean(q90 > q95)

print(f"Crossing P50 > P90: {crossing_50_90:.2%}")
print(f"Crossing P90 > P95: {crossing_90_95:.2%}")



## 11. Teste final

**Só rode esta seção depois que:**

- as features estiverem congeladas;
- o modelo pontual estiver escolhido;
- os parâmetros do modelo estiverem escolhidos;
- a estratégia quantílica estiver definida.

Nesta etapa, o teste final deixa de ser ferramenta de desenvolvimento e passa a ser a evidência final do TCC.


In [ ]:

# Escolha automática do melhor modelo de ML pelo MAE de validação.
# Você pode substituir manualmente se a análise de RMSE/cauda justificar outra escolha.
ml_candidates = model_comparison[
    ~model_comparison["model"].str.startswith("baseline")
].sort_values("mae")

BEST_POINT_MODEL_NAME = ml_candidates.iloc[0]["model"]
print("Modelo pontual selecionado:", BEST_POINT_MODEL_NAME)

final_train_df = pd.concat([train_df, val_df, cal_df], ignore_index=True)

X_final_train = final_train_df[FEATURES]
y_final_train = final_train_df[TARGET]
X_test = test_df[FEATURES]
y_test = test_df[TARGET]

if BEST_POINT_MODEL_NAME == "ridge_log_target":
    final_point_model = clone(ridge)
    final_point_model.fit(X_final_train, np.log1p(y_final_train))
    pred_test = np.expm1(final_point_model.predict(X_test))

elif BEST_POINT_MODEL_NAME == "random_forest":
    final_point_model = clone(rf)
    final_point_model.fit(X_final_train, y_final_train)
    pred_test = final_point_model.predict(X_test)

elif BEST_POINT_MODEL_NAME == "gradient_boosting":
    final_point_model = clone(gbr)
    final_point_model.fit(X_final_train, y_final_train)
    pred_test = final_point_model.predict(X_test)

else:
    raise ValueError(
        f"Modelo não tratado no bloco final: {BEST_POINT_MODEL_NAME}"
    )

pred_test = np.clip(pred_test, 0, None)

final_point_metrics = regression_metrics(y_test, pred_test)
final_point_metrics



### 11.1 Quantis finais


In [ ]:

final_quantile_models = {}
final_quantile_predictions = {}

for q in QUANTILES:
    model = Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("model", GradientBoostingRegressor(
            loss="quantile",
            alpha=q,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        )),
    ])

    model.fit(X_final_train, y_final_train)
    final_quantile_models[q] = model
    final_quantile_predictions[q] = np.clip(model.predict(X_test), 0, None)

final_quantile_eval = []

for q in QUANTILES:
    pred = final_quantile_predictions[q]
    final_quantile_eval.append({
        "quantile": q,
        "pinball_loss": mean_pinball_loss(y_test, pred, alpha=q),
        "empirical_coverage": np.mean(y_test.to_numpy() <= pred),
        "coverage_error": np.mean(y_test.to_numpy() <= pred) - q,
    })

pd.DataFrame(final_quantile_eval)



## 12. Interpretabilidade

Primeiro use uma ferramenta simples: importância das variáveis no modelo de árvore.

Se o modelo final for baseado em árvores, depois podemos acrescentar SHAP como evidência complementar.

A pergunta do TCC não é “qual feature tem o maior número?”, mas:

> A importância observada faz sentido em relação ao EDA? Porto, perfil operacional, calendário, clima e congestionamento continuam aparecendo como drivers relevantes?


In [ ]:

preprocessor = final_point_model.named_steps["preprocessor"]
fitted_model = final_point_model.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()

if hasattr(fitted_model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": fitted_model.feature_importances_,
    }).sort_values("importance", ascending=False)

elif hasattr(fitted_model, "coef_"):
    coefs = np.ravel(fitted_model.coef_)
    feature_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": np.abs(coefs),
        "signed_coefficient": coefs,
    }).sort_values("importance", ascending=False)

else:
    raise ValueError("Modelo final sem importância ou coeficientes diretamente disponíveis.")

feature_importance.head(30)


In [ ]:

top_imp = feature_importance.head(20).sort_values("importance")

plt.figure(figsize=(9, 7))
plt.barh(top_imp["feature"], top_imp["importance"])
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.title("Top 20 variáveis do modelo")
plt.tight_layout()
plt.show()



## 13. Ponte para estoque de segurança e capital de giro

### Limitação importante

A base do Porto Sem Papel não contém:

- demanda do item;
- custo unitário;
- estoque real;
- lead time completo fornecedor → cliente.

Logo, este TCC **não pode afirmar uma redução real em R$ observada numa empresa**.

O que podemos fazer de forma metodologicamente correta é um **exercício de cenário / sensibilidade**.

### Ideia

Usamos P50 e P90 previstos para representar uma distribuição condicional de lead time.

Como o lead time é positivo e assimétrico, faremos uma aproximação lognormal:

- P50 determina a localização;
- P90 determina a dispersão da cauda.

Depois calculamos:

\[
E[D(L)] = \mu_D E[L]
\]

\[
Var(D(L)) = \sigma_D^2 E[L] + \mu_D^2 Var(L)
\]

\[
SS = k \sqrt{Var(D(L))}
\]

Os parâmetros de demanda abaixo são **hipotéticos e editáveis**.


In [ ]:

from scipy.stats import norm

def lognormal_moments_from_q50_q90(q50_days, q90_days):
    eps = 1e-6
    q50_days = np.clip(np.asarray(q50_days), eps, None)
    q90_days = np.clip(np.asarray(q90_days), q50_days + eps, None)

    z90 = norm.ppf(0.90)

    mu_log = np.log(q50_days)
    sigma_log = (np.log(q90_days) - np.log(q50_days)) / z90
    sigma_log = np.clip(sigma_log, 0, None)

    mean_l = np.exp(mu_log + 0.5 * sigma_log**2)
    var_l = (np.exp(sigma_log**2) - 1) * np.exp(
        2 * mu_log + sigma_log**2
    )

    return mean_l, var_l

def safety_stock_units(mean_lead_days, var_lead_days, demand_mean_day, demand_std_day, service_z):
    variance_during_lead = (
        demand_std_day**2 * mean_lead_days
        + demand_mean_day**2 * var_lead_days
    )
    return service_z * np.sqrt(np.maximum(variance_during_lead, 0))



### 13.1 Política dinâmica baseada nos quantis previstos

Primeiro, convertemos as previsões de horas para dias.


In [ ]:

dynamic_q50_days = final_quantile_predictions[0.50] / 24
dynamic_q90_days = final_quantile_predictions[0.90] / 24

dyn_mean_l, dyn_var_l = lognormal_moments_from_q50_q90(
    dynamic_q50_days,
    dynamic_q90_days,
)

DEMAND_MEAN_PER_DAY = 100.0
DEMAND_STD_PER_DAY = 20.0
SERVICE_Z = 1.645   # cenário ilustrativo de 95%
UNIT_COST = 1.0     # capital normalizado; troque por custo real se houver

dynamic_ss = safety_stock_units(
    dyn_mean_l,
    dyn_var_l,
    DEMAND_MEAN_PER_DAY,
    DEMAND_STD_PER_DAY,
    SERVICE_Z,
)



### 13.2 Benchmark estático

Como não temos o parâmetro de ERP de uma empresa real, vamos construir um benchmark histórico:

- P50 histórico por porto;
- P90 histórico por porto;
- fallback global para portos sem histórico suficiente.

Isso é mais justo do que comparar o ML somente contra um único lead time global.


In [ ]:

def historical_quantile_by_group(train, scoring, group_col, q, min_group_size=30):
    global_q = train[TARGET].quantile(q)

    stats = (
        train.groupby(group_col, dropna=False)[TARGET]
        .agg(
            quantile=lambda s: s.quantile(q),
            count="size",
        )
        .reset_index()
    )

    stats.loc[stats["count"] < min_group_size, "quantile"] = np.nan

    out = scoring[[group_col]].merge(
        stats[[group_col, "quantile"]],
        on=group_col,
        how="left",
    )

    return out["quantile"].fillna(global_q).to_numpy()

historical_reference = final_train_df.copy()

static_q50_days = (
    historical_quantile_by_group(
        historical_reference,
        test_df,
        "port",
        q=0.50,
        min_group_size=30,
    ) / 24
)

static_q90_days = (
    historical_quantile_by_group(
        historical_reference,
        test_df,
        "port",
        q=0.90,
        min_group_size=30,
    ) / 24
)

sta_mean_l, sta_var_l = lognormal_moments_from_q50_q90(
    static_q50_days,
    static_q90_days,
)

static_ss = safety_stock_units(
    sta_mean_l,
    sta_var_l,
    DEMAND_MEAN_PER_DAY,
    DEMAND_STD_PER_DAY,
    SERVICE_Z,
)



### 13.3 Comparação de cenário

O resultado abaixo deve ser escrito como:

- “redução simulada”;
- “cenário normalizado”;
- “sob as premissas de demanda e serviço adotadas”.

Não escreva como economia real observada.


In [ ]:

scenario = pd.DataFrame({
    "static_ss_units": static_ss,
    "dynamic_ss_units": dynamic_ss,
})

scenario["ss_change_units"] = (
    scenario["dynamic_ss_units"] - scenario["static_ss_units"]
)

scenario["capital_static"] = scenario["static_ss_units"] * UNIT_COST
scenario["capital_dynamic"] = scenario["dynamic_ss_units"] * UNIT_COST

summary_scenario = pd.DataFrame({
    "metric": [
        "SS médio estático",
        "SS médio dinâmico",
        "Variação média de SS",
        "Variação percentual do capital normalizado",
    ],
    "value": [
        scenario["static_ss_units"].mean(),
        scenario["dynamic_ss_units"].mean(),
        scenario["ss_change_units"].mean(),
        (
            scenario["capital_dynamic"].mean()
            / scenario["capital_static"].mean()
            - 1
        ) * 100,
    ],
})

summary_scenario



### 13.4 Não force o resultado

É possível que a política dinâmica:

- reduza estoque em contextos previsíveis;
- aumente estoque em contextos de alto risco.

Isso não é um problema.

O valor do modelo pode estar justamente em **realocar o buffer**, reduzindo excesso onde o risco é baixo e protegendo onde o risco é alto.


In [ ]:

scenario["direction"] = np.where(
    scenario["dynamic_ss_units"] < scenario["static_ss_units"],
    "reduz",
    "aumenta",
)

scenario["direction"].value_counts(normalize=True).rename("proporção")



## 14. Tabelas finais para o Capítulo 4

Depois que o experimento estiver congelado, salve:

1. comparação dos modelos;
2. métricas quantílicas;
3. importância das variáveis;
4. cenário de estoque/capital;
5. previsões finais para auditoria.


In [ ]:

model_comparison.to_csv(
    OUTPUT_DIR / "chapter4_model_comparison_validation.csv",
    index=False,
)

pd.DataFrame(final_quantile_eval).to_csv(
    OUTPUT_DIR / "chapter4_quantile_metrics_final.csv",
    index=False,
)

feature_importance.to_csv(
    OUTPUT_DIR / "chapter4_feature_importance.csv",
    index=False,
)

summary_scenario.to_csv(
    OUTPUT_DIR / "chapter4_safety_stock_scenario.csv",
    index=False,
)

final_predictions = test_df[
    ["port_call_id", DATE_COL, "port", "region", "state", "operation_type", TARGET]
].copy()

final_predictions["pred_point"] = pred_test
final_predictions["pred_p50"] = final_quantile_predictions[0.50]
final_predictions["pred_p90"] = final_quantile_predictions[0.90]
final_predictions["pred_p95"] = final_quantile_predictions[0.95]

final_predictions.to_parquet(
    OUTPUT_DIR / "chapter4_final_predictions.parquet",
    index=False,
)

print("Arquivos do Capítulo 4 salvos.")



# 15. Como transformar os resultados em texto do TCC

Depois de executar tudo, o Capítulo 4 pode seguir esta estrutura:

## 4.1 Desenho experimental e prevenção de vazamento temporal
- instante da previsão;
- features disponíveis;
- divisão treino/validação/calibração/teste;
- métricas.

## 4.2 Baselines históricos
- mediana global;
- mediana por porto;
- baseline hierárquico porto + operação;
- ganho percentual.

## 4.3 Modelos de regressão
- Ridge;
- Random Forest;
- Gradient Boosting;
- comparação na validação;
- escolha final.

## 4.4 Avaliação no teste final
- MAE, RMSE, MedAE e RMSLE;
- ganho versus melhor baseline;
- análise por região, porto e faixa de permanência.

## 4.5 Modelagem do risco de cauda
- P50, P90, P95;
- pinball loss;
- cobertura empírica;
- quantile crossing.

## 4.6 Interpretabilidade
- importância das variáveis;
- relação com os drivers identificados no EDA;
- SHAP opcional para o modelo final.

## 4.7 Simulação de estoque de segurança e capital de giro
- premissas;
- benchmark estático;
- política dinâmica;
- resultado médio e dispersão;
- enfatizar que é um cenário, não economia real observada.

## 4.8 Discussão e limitações
- permanência portuária é proxy de lead time;
- ausência de demanda/custo reais;
- variáveis climáticas observadas vs informação disponível na previsão;
- limitações de generalização;
- próximos passos.

# 16. Checklist de entendimento

Antes de considerar o Capítulo 4 concluído, responda sem consultar o código:

1. Qual é o target?
2. Em que instante a previsão é feita?
3. Por que não usamos `arrivals_same_day_port`?
4. Por que treino e teste não podem ser embaralhados?
5. Qual a diferença entre MAE, RMSE e MedAE?
6. O que a mediana por porto representa?
7. Por que um grupo `porto + operação` com 1 observação é perigoso?
8. Por que o Ridge usa escala log?
9. Por que modelos de árvore são úteis neste problema?
10. O que significa uma previsão P90?
11. Como você verifica se o P90 está calibrado?
12. Como P50/P90 se conectam ao exercício de estoque?
13. Por que o resultado financeiro deve ser chamado de simulação?
14. Quais conclusões são suportadas pelos dados e quais são apenas hipóteses?
